# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a thorough guide for loading, exploring, and processing the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which precisely describes the structure and semantics of available data and metadata.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset's Croissant URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Access dataset metadata as Python dict for convenient pretty-printing
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata.get('name', '')}\n\nDescription: {metadata.get('description', '')}\n")

## 2. Data Overview
Let’s review available record sets, fields, and their `@id`s as described in the Croissant schema, using the dataset metadata.

**Note:** All references to record sets, fields, and columns will use their `@id` for precise programmatic access.

In [ ]:
# List all record sets with their @id and name
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- record_set @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '(no name)')}")
    # List fields in this record set
    if 'field' in rs:
        print("  fields:")
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"    - {field['@id']}: {field.get('name','')}")
            else:
                print(f"    - {field}")
    print()

## 3. Data Extraction
Let us select one or more record sets to load their contents, and represent their records as DataFrames for further analysis.

> For this dataset, the main tabular data is typically found in the primary `RecordSet`. Use the `@id` of the desired record set from the overview above.

In [ ]:
# Identify the main record set(s) @id to extract
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Iterate all records for current record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} into DataFrame with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set {record_set_id}.")

In [ ]:
# For demonstration, select the first non-empty record set
main_record_set_id = None
for k in dataframes.keys():
    if not dataframes[k].empty:
        main_record_set_id = k
        break
if main_record_set_id is None:
    raise RuntimeError('No records sets could be loaded as DataFrames.')

print(f"Column names for record set {main_record_set_id}:\n")
print(dataframes[main_record_set_id].columns.tolist())

print(f"\nFirst 5 records:\n")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze a numeric field, as an example of data preparation and summarization.

Operations here include filtering on a threshold, normalizing numerical data, and grouping by a categorical field, all using field `@id`.

In [ ]:
# Choose a numeric field to analyze.
# List numeric columns and pick one (e.g., peptides_count)
possible_numeric_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in ['int64','float64']]
print(f"Possible numeric fields in {main_record_set_id}: {possible_numeric_fields}")
numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else None
if numeric_field_id is None:
    raise RuntimeError('No numeric columns identified for analysis.')
print(f"Selected numeric field for analysis: {numeric_field_id}")

# Filter records with values above a threshold
threshold = dataframes[main_record_set_id][numeric_field_id].mean()
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
print(f"Filtered records in {main_record_set_id} where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows\n")
display(filtered_df.head())

# Normalize the numeric field
col_name_norm = f"{numeric_field_id}_normalized"
filtered_df[col_name_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nFirst 5 normalized {numeric_field_id} values:")
display(filtered_df[[numeric_field_id, col_name_norm]].head())

# Attempt to group by a categorical field
possible_group_fields = [col for col in filtered_df.columns if filtered_df[col].dtype == 'object']
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(f"Grouped data by {group_field} (showing top 5 groups):")
    display(grouped_df.head(5))
else:
    print("No categorical field available for group-by demonstration.")

## 5. Visualization
We visualize the numeric field distribution and, if grouping is possible, show mean values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping exists, plot mean value per group
if group_field:
    plt.figure(figsize=(8,4))
    sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and interrogate the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` Python library.

- We listed record sets and their fields using `@id` for reliable reference.
- Data was extracted from the primary record set and represented as a DataFrame.
- Typical exploratory techniques were demonstrated: filtering, normalization, grouping, and plotting.

**Next steps:**
- Deeper domain analysis by mapping field semantics to biological questions
- Integration with external controlled vocabularies
- Building machine learning or statistical analysis pipelines

_For more details on the dataset structure and Croissant schema, [see the FAIR² metadata page here](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)._